In [188]:
import pandas as pd

data = pd.read_csv("home_price.csv")

In [189]:
data[['Fiyat', 'Net_Metrekare']].describe()

,Fiyat,Net_Metrekare
count,2.032600e+04,20326.000000
mean,4.649510e+06,151.964528
std,7.136120e+07,1137.304466
min,2.000000e+04,0.000000
25%,1.640000e+06,90.000000
50%,2.300000e+06,120.000000
75%,3.300000e+06,150.000000
max,7.500000e+09,110000.000000


In [190]:
data = data.drop(["Tapu_Durumu", "Takas", "Yatırıma_Uygunluk", "Eşya_Durumu"], axis=1)
 

In [191]:
data.head()

,Net_Metrekare,Brüt_Metrekare,Oda_Sayısı,Bulunduğu_Kat,Binanın_Yaşı,Isıtma_Tipi,Fiyat,Şehir,Binanın_Kat_Sayısı,Kullanım_Durumu,Banyo_Sayısı
0,120,150.0,4.0,4.Kat,21 Ve Üzeri,Kombi Doğalgaz,950000.0,adana,6,Boş,1.0
1,100,125.0,4.0,3.Kat,4,Kombi Doğalgaz,1250000.0,adana,10,Boş,1.0
2,89,95.0,3.0,4.Kat,0 (Yeni),Kombi Doğalgaz,1750000.0,adana,14,Boş,1.0
3,40,55.0,2.0,6.Kat,0 (Yeni),Kombi Doğalgaz,1300000.0,adana,14,Boş,1.0
4,140,150.0,4.0,Düz Giriş (Zemin),5-10,Klimalı,1700000.0,adana,4,Boş,1.0


In [192]:
data = data.drop_duplicates(keep='first')

In [193]:
data = data.dropna()

In [194]:
# 1. Domain filtering (mantık kontrolü)
data = data[data["Brüt_Metrekare"] >= data["Net_Metrekare"]]

data = data[(data["Oda_Sayısı"] >= 1) & (data["Oda_Sayısı"] <= 10)]
data = data[(data["Banyo_Sayısı"] >= 1) & (data["Banyo_Sayısı"] <= 6)]


# Net metrekareyi 30 ile 500 arası, fiyatı ise 500 bin ile 50 milyon arası sınırlama
data = data[(data['Net_Metrekare'] >= 30) & (data['Net_Metrekare'] <= 300)]
data = data[(data['Fiyat'] >= 500000) & (data['Fiyat'] <= 10000000)]

# # 2. Statistical filtering (outlier)
# for col in ["Fiyat", "Net_Metrekare"]:
#     lower = data[col].quantile(0.01)
#     upper = data[col].quantile(0.99)

#     data = data[(data[col] >= lower) & (data[col] <= upper)]


In [195]:


data['Bulunduğu_Kat'].unique()

array(['4.Kat', '3.Kat', '6.Kat', 'Düz Giriş (Zemin)', '12.Kat', '2.Kat',
       '8.Kat', '5.Kat', '14.Kat', '1.Kat', '17.Kat', '9.Kat',
       'Yüksek Giriş', '7.Kat', '11.Kat', 'Müstakil', 'Bahçe Katı',
       '10.Kat', '15.Kat', 'Bahçe Dublex', '13.Kat', 'Kot 3 (-3).Kat',
       '18.Kat', 'Çatı Dubleks', 'Kot 2 (-2).Kat', 'Bodrum Kat', '16.Kat',
       'Çatı Katı', 'Villa Tipi', '26.Kat', 'Kot 1 (-1).Kat',
       'Kot 4 (-4).Kat', '21.Kat', '40+.Kat', '19.Kat', '30.Kat',
       '22.Kat'], dtype=object)

##### Elimizde şehirler verisi var bunu ohe encode ettiğimizde çok fazla sütun ortaya çıkıyor bunu engellemek için ilk önce map ile kat değerlerini sayısal değerlere çevirip fiyat ile korelasyon değerine bakacağım sonra ona göre kat bilgisini sadece villamı müstakil evmi şeklinde sütunlar oluşturacağım diğer detaylı kat bilgisini direkt sileceğeim tabi bunlar korelasyon değeri düşükse yapacağım

In [196]:
data["Apartman_Mi"] = 0
data["Zemin_Mi"] = 0
data["Cati_Mi"] = 0

for i in range(len(data)):
    kat = str(data.iloc[i]["Bulunduğu_Kat"]).lower()
    
    if "villa" not in kat and "müstakil" not in kat:
        data.loc[i, "Apartman_Mi"] = 1
        
    if "zemin" in kat or "bahçe" in kat:
        data.loc[i, "Zemin_Mi"] = 1
        
    if "çatı" in kat:
        data.loc[i, "Cati_Mi"] = 1




##### Veriye yeni sütunlar ekledik Apartman_Mi , Zemin_Mi,Cati_Mi olarak 

In [197]:
data.drop("Bulunduğu_Kat", axis=1, inplace=True)

- String ve karmaşık veriyi kaldırdın.
- Onun yerine oluşturduğun yeni sütunlar (Apartman_Mi vs.) kullanılacak.

In [198]:
data.head()

,Net_Metrekare,Brüt_Metrekare,Oda_Sayısı,Binanın_Yaşı,Isıtma_Tipi,Fiyat,Şehir,Binanın_Kat_Sayısı,Kullanım_Durumu,Banyo_Sayısı,Apartman_Mi,Zemin_Mi,Cati_Mi
0,120.0,150.0,4.0,21 Ve Üzeri,Kombi Doğalgaz,950000.0,adana,6.0,Boş,1.0,1.0,0.0,0.0
1,100.0,125.0,4.0,4,Kombi Doğalgaz,1250000.0,adana,10.0,Boş,1.0,1.0,0.0,0.0
2,89.0,95.0,3.0,0 (Yeni),Kombi Doğalgaz,1750000.0,adana,14.0,Boş,1.0,1.0,0.0,0.0
3,40.0,55.0,2.0,0 (Yeni),Kombi Doğalgaz,1300000.0,adana,14.0,Boş,1.0,1.0,0.0,0.0
4,140.0,150.0,4.0,5-10,Klimalı,1700000.0,adana,4.0,Boş,1.0,1.0,1.0,0.0


In [199]:
yas_map = {
    "0 (Yeni)": 0,
    "1-5": 3,
    "5-10": 7,
    "11-15": 13,
    "16-20": 18,
    "21 Ve Üzeri": 25
}

data["Binanın_Yaşı"] = data["Binanın_Yaşı"].map(yas_map)

- Binanın yaşı metin olduğu için sayıya çevirdin.
- Model artık “yaş arttıkça ne olur” ilişkisini öğrenebilir.

In [200]:
data = pd.get_dummies(data, columns=["Isıtma_Tipi"], drop_first=True)

- Isıtma tiplerini ayrı sütunlara böldün (one-hot encoding).
- Model, her ısıtma tipinin fiyat üzerindeki etkisini ayrı ayrı öğrenir.

In [201]:
kullanim_durumu_map = {
    "Boş": 0,
    "Kiracılı": 1
}
data["Kullanım_Durumu"] = data["Kullanım_Durumu"].map(kullanim_durumu_map)

- “Boş / Kiracılı” bilgisini 0-1 yaptın.
- Model bunu basit bir durum (var/yok) olarak kullanır.

In [202]:
data["Şehir"] = data.groupby("Şehir")["Fiyat"].transform("mean")

- Her şehri, o şehirdeki ortalama fiyatla değiştirdin.
- Model için çok güçlü bir sinyal: “bu şehir pahalı mı ucuz mu”

In [203]:
data.head()

,Net_Metrekare,Brüt_Metrekare,Oda_Sayısı,Binanın_Yaşı,Fiyat,Şehir,Binanın_Kat_Sayısı,Kullanım_Durumu,Banyo_Sayısı,Apartman_Mi,...,Isıtma_Tipi_Isıtma Yok,Isıtma_Tipi_Jeotermal,Isıtma_Tipi_Kat Kaloriferi,Isıtma_Tipi_Klimalı,Isıtma_Tipi_Kombi Doğalgaz,Isıtma_Tipi_Merkezi (Pay Ölçer),Isıtma_Tipi_Merkezi Doğalgaz,Isıtma_Tipi_Merkezi Kömür,Isıtma_Tipi_Sobalı,Isıtma_Tipi_Yerden Isıtma
0,120.0,150.0,4.0,25.0,950000.0,2.459322e+06,6.0,0.0,1.0,1.0,...,False,False,False,False,True,False,False,False,False,False
1,100.0,125.0,4.0,NaN,1250000.0,2.459322e+06,10.0,0.0,1.0,1.0,...,False,False,False,False,True,False,False,False,False,False
2,89.0,95.0,3.0,0.0,1750000.0,2.459322e+06,14.0,0.0,1.0,1.0,...,False,False,False,False,True,False,False,False,False,False
3,40.0,55.0,2.0,0.0,1300000.0,2.459322e+06,14.0,0.0,1.0,1.0,...,False,False,False,False,True,False,False,False,False,False
4,140.0,150.0,4.0,7.0,1700000.0,2.459322e+06,4.0,0.0,1.0,1.0,...,False,False,False,True,False,False,False,False,False,False


In [204]:
data['Fiyat'].max()

np.float64(10000000.0)

In [205]:
data["Fiyat"].describe()

count    1.720100e+04
mean     2.473925e+06
std      1.309284e+06
min      5.000000e+05
25%      1.600000e+06
50%      2.200000e+06
75%      3.000000e+06
max      1.000000e+07
Name: Fiyat, dtype: float64

In [206]:
data.isnull().sum()

Net_Metrekare                      2563
Brüt_Metrekare                     2563
Oda_Sayısı                         2563
Binanın_Yaşı                       5749
Fiyat                              2563
Şehir                              2563
Binanın_Kat_Sayısı                 2563
Kullanım_Durumu                    9196
Banyo_Sayısı                       2563
Apartman_Mi                           0
Zemin_Mi                           2389
Cati_Mi                            2516
Isıtma_Tipi_Doğalgaz Sobalı           0
Isıtma_Tipi_Güneş Enerjisi            0
Isıtma_Tipi_Isıtma Yok                0
Isıtma_Tipi_Jeotermal                 0
Isıtma_Tipi_Kat Kaloriferi            0
Isıtma_Tipi_Klimalı                   0
Isıtma_Tipi_Kombi Doğalgaz            0
Isıtma_Tipi_Merkezi (Pay Ölçer)       0
Isıtma_Tipi_Merkezi Doğalgaz          0
Isıtma_Tipi_Merkezi Kömür             0
Isıtma_Tipi_Sobalı                    0
Isıtma_Tipi_Yerden Isıtma             0
dtype: int64

In [207]:
# target
data = data.dropna(subset=["Fiyat"])

# numeric
num_cols = ["Net_Metrekare","Brüt_Metrekare","Oda_Sayısı","Banyo_Sayısı","Binanın_Kat_Sayısı","Binanın_Yaşı"]
for col in num_cols:
    data[col].fillna(data[col].median(), inplace=True)

# binary
data["Kullanım_Durumu"].fillna(0, inplace=True)

# şehir
data["Şehir"].fillna(data["Şehir"].mean(), inplace=True)

C:\Users\metin\AppData\Local\Temp\ipykernel_14964\1837930212.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)
C:\Users\metin\AppData\Local\Temp\ipykernel_14964\1837930212.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

 - Fiyat (hedef değişken) eksik olan satırları sildik, çünkü model hedef değeri olmadan öğrenemez.
 - Diğer sütunlardaki eksik değerleri ise veri tipine göre doldurduk: sayısalları median ile, binary’yi 0 ile, şehir bilgisini ortalama değerle tamamladık.

In [209]:
data.isnull().sum()

Net_Metrekare                      0
Brüt_Metrekare                     0
Oda_Sayısı                         0
Binanın_Yaşı                       0
Fiyat                              0
Şehir                              0
Binanın_Kat_Sayısı                 0
Kullanım_Durumu                    0
Banyo_Sayısı                       0
Apartman_Mi                        0
Zemin_Mi                           0
Cati_Mi                            0
Isıtma_Tipi_Doğalgaz Sobalı        0
Isıtma_Tipi_Güneş Enerjisi         0
Isıtma_Tipi_Isıtma Yok             0
Isıtma_Tipi_Jeotermal              0
Isıtma_Tipi_Kat Kaloriferi         0
Isıtma_Tipi_Klimalı                0
Isıtma_Tipi_Kombi Doğalgaz         0
Isıtma_Tipi_Merkezi (Pay Ölçer)    0
Isıtma_Tipi_Merkezi Doğalgaz       0
Isıtma_Tipi_Merkezi Kömür          0
Isıtma_Tipi_Sobalı                 0
Isıtma_Tipi_Yerden Isıtma          0
dtype: int64

In [210]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17201 entries, 0 to 20322
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Net_Metrekare                    17201 non-null  float64
 1   Brüt_Metrekare                   17201 non-null  float64
 2   Oda_Sayısı                       17201 non-null  float64
 3   Binanın_Yaşı                     17201 non-null  float64
 4   Fiyat                            17201 non-null  float64
 5   Şehir                            17201 non-null  float64
 6   Binanın_Kat_Sayısı               17201 non-null  float64
 7   Kullanım_Durumu                  17201 non-null  float64
 8   Banyo_Sayısı                     17201 non-null  float64
 9   Apartman_Mi                      17201 non-null  float64
 10  Zemin_Mi                         17201 non-null  float64
 11  Cati_Mi                          17201 non-null  float64
 12  Isıtma_Tipi_Doğalgaz So

In [208]:
data.to_csv("clean_data.csv", index=False)